# Customer Churn Prediction — ML Pipeline (Notebook 3 of 4)

**Goal of this notebook:** Build a single, reusable pipeline that handles preprocessing, class imbalance, and model training together — the way this would be done in production.
**Why:** Doing scaling/encoding/SMOTE manually and separately risks data leakage (e.g. scaling using test data statistics). Wrapping everything in one `Pipeline` object guarantees every step is applied identically and correctly to train and test data.
**Input:** `../data/processed/telco_clean.csv` (saved in notebook 2)
**Output:** trained pipeline + train/test splits used by notebook 4

In [2]:
import pandas as pd
import os

df = pd.read_csv("../data/processed/telco_clean.csv")
df.shape

(7043, 22)

In [3]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket,services_count
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-12,1
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,0,25-48,2
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-12,2
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,25-48,3
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,0-12,0


In [7]:
df['Churn'].value_counts(normalize=True) # confirm imbalance carried over correctly

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

### Define feature groups
Splitting columns into numeric and categorical groups because each needs different preprocessing — numeric columns get scaled, categorical columns get one-hot encoded.

In [8]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'services_count']

categorical_features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_bucket'
]

len(numeric_features), len(categorical_features)

(4, 17)

### Build preprocessing pipelines
Numeric: impute any missing values with median, then scale (important for models sensitive to feature magnitude, and good practice generally).
Categorical: impute missing with most frequent value, then one-hot encode. `handle_unknown='ignore'` prevents errors if test data has a category not seen in training.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

### Handle class imbalance with SMOTE
Churn is ~26% of the data. Without correcting this, the model can get high accuracy by mostly predicting "no churn" — which is useless for the business goal. SMOTE generates synthetic examples of the minority class so the model learns to recognize churners properly. It must go inside the pipeline (using imblearn's Pipeline) so it's only applied to training folds, never to test data.

In [11]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

### Train/test split
`stratify=y` keeps the same churn ratio in both train and test sets — important since churn is imbalanced, otherwise a random split could give a test set with very few churners.

In [12]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape

((5634, 21), (1409, 21))

### Fit the baseline pipeline

In [14]:
pipeline.fit(X_train,y_train)

C:\Users\Roshi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Roshi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\Roshi\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Roshi\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges',
                                                   'services_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod',
                                                   'tenure_bucket'])])),
                ('smote', SMOTE(random_state=42)),
                ('classifier', RandomForestClassifier(random_state=42))])

### Compare against other models
Random Forest is the primary model, but comparing against Logistic Regression (simple baseline) and XGBoost (often stronger on tabular data) shows the choice was justified, not arbitrary.

In [15]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

from sklearn.metrics import roc_auc_score

for name, clf in models.items():
    pipe = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    print(f"{name}: ROC-AUC = {roc_auc_score(y_test, proba):.4f}")

Logistic Regression: ROC-AUC = 0.8390
Random Forest: ROC-AUC = 0.8195
XGBoost: ROC-AUC = 0.8184


### Hyperparameter tuning
Using GridSearchCV to find better settings for the chosen model instead of guessing. Scoring on ROC-AUC (not accuracy) because the data is imbalanced.

In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

grid.best_params_

Fitting 3 folds for each of 18 candidates, totalling 54 fits


{'classifier__max_depth': 10,
 'classifier__min_samples_leaf': 4,
 'classifier__n_estimators': 200}

In [19]:
best_pipeline = grid.best_estimator_
grid.best_score_

np.float64(0.8439493321447052)

### Save the trained pipeline
Saving the fitted pipeline so notebook 4 (evaluation) doesn't need to retrain — it just loads this and evaluates it.

In [20]:
import joblib
import os

os.makedirs("../outputs", exist_ok=True)
joblib.dump(best_pipeline, "../outputs/churn_pipeline.pkl")

# Also save train/test splits so notebook 4 can load them directly
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved pipeline and train/test splits.")

Saved pipeline and train/test splits.


## Summary

- Built a single `Pipeline` combining preprocessing (scaling + one-hot encoding), SMOTE (class imbalance handling), and a classifier
- Split data with stratification to preserve churn ratio in train/test sets
- Compared Logistic Regression, Random Forest, and XGBoost using ROC-AUC
- Tuned the best-performing model with `GridSearchCV`
- Saved the trained pipeline (`churn_pipeline.pkl`) and train/test splits for the next notebook

**Next step:** Notebook 04 — evaluate the model properly (classification report, ROC-AUC, confusion matrix, SHAP feature importance) and export predictions for Power BI.